In [ ]:
# !pip install -q pymorphy3

In [ ]:
import re
import string
from collections import Counter
from functools import cache, partial

import nltk
import numpy as np
import pandas as pd
import pymorphy3
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# 1. Загружаем корпус и объединяем

In [ ]:
pos = pd.read_csv("/content/positive.csv", sep=";", header=None)
neg = pd.read_csv("/content/negative.csv", sep=";", header=None)

In [ ]:
tweet = pd.concat([pos, neg])
tweet.head(5)

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906692374446080,1386325927,pleease_shut_up,"@first_timee хоть я и школота, но поверь, у на...",1,0,0,0,7569,62,61,0
1,408906692693221377,1386325927,alinakirpicheva,"Да, все-таки он немного похож на него. Но мой ...",1,0,0,0,11825,59,31,2
2,408906695083954177,1386325927,EvgeshaRe,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,0,1,0,1273,26,27,0
3,408906695356973056,1386325927,ikonnikova_21,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,0,1,0,1549,19,17,0
4,408906761416867842,1386325943,JumpyAlex,@irina_dyshkant Вот что значит страшилка :D\nН...,1,0,0,0,597,16,23,1


# 2. Токенизируем и удаляем стоп-слова и пунктуацию

In [ ]:
def clean_text(text: str, pattern: str):
    # тэги, хэштэги, ссылки
    text = re.sub(pattern, "", text)
    text = re.sub(r"\s+", " ", text).strip()

    # смех
    text = re.sub(r"(ах){2,}", r"ахах", text)
    text = re.sub(r"(ха){2,}", r"ха", text)

    text = re.sub(r"(\w)\1+", r"\1", text)  # блиин -> блин

    text = re.sub(r"(^|\s)[^\w\s]+(\w)", r"\1\2", text)  # -слушай -> слушай
    text = re.sub(r"(\w)[^\w\s]+($|\s)", r"\1\2", text)  # человек… -> человек
    return text


def remove_stopwords(text: str, stop_words: set):
    return [word for word in text if word not in stop_words]


def cleaning(text: str, stop_words: set, pattern: str = r"@\w+|https?://\S+|#\w+|™"):
    smiles = {"d": "ахах", "p": "ахах"}
    cleaned = clean_text(text.lower().strip(), pattern)
    tokenized = [smiles.get(word, word).lower() for word in word_tokenize(cleaned)]

    return remove_stopwords(tokenized, stop_words)


bad = {"типа", "кстати", "вроде", "бля", "блять", "бл", "пиздец", "похуй", "хуйню", "та"}
elipsis = {"...", "…"}
tweet_stuff = {"rt"}
keep = {
    "и",
    "больше",
    "вас",
    "вам",
    "его",
    "ее",
    "ей",
    "ему",
    "им",
    "лучше",
    "мой",
    "моя",
    "мы",
    "нас",
    "не",
    "нет",
    "ни",
    "сам",
    "себе",
    "себя",
    "ты",
    "два",
    "три",
    "хорошо",
    "эти",
    "тебя",
    "тебе",
    "для",
}
stop_words = set(stopwords.words("russian")) - keep | set(string.punctuation) | tweet_stuff | elipsis | bad

part_func = partial(cleaning, stop_words=stop_words)
sentences = tweet[3].apply(part_func).to_list()
sentences[0]

['и',
 'школота',
 'поверь',
 'нас',
 'самое',
 'ахах',
 'общество',
 'профилирующий',
 'предмет']

# 3. Загружаем словарь тональностей, делаем предобработку и получаем вектор тональностей

In [ ]:
lang_feat = pd.read_csv("/content/words_all_full_rating.csv", sep=";", encoding="cp1251")
lang_feat = lang_feat.dropna(axis="columns")
lang_feat["mean"] = lang_feat["mean"].str.replace(",", ".").astype(float)
lang_feat

,Words,mean,dispersion,average rate
0,аборигенный,-0.250000,"0,433012701892219",0
1,аборт,-1.000000,"0,816496580927726",-1
2,абрамович,0.000000,0,0
3,абсолютный,0.333333,"0,471404520791032",0
4,абстрактный,-0.111111,"0,87488976377909",0
...,...,...,...,...
7540,ярый,-0.333333,"0,942809041582063",0
7541,ясно,0.000000,0,0
7542,ясность,0.666667,"0,471404520791032",1
7543,ясный,0.666667,"0,471404520791032",1


In [ ]:
@cache
def get_lemma(word: str, moprh: pymorphy3.MorphAnalyzer) -> str:
    return morph.parse(word)[0].normal_form


def lemmatize(text: list[str], moprh: pymorphy3.MorphAnalyzer) -> list[str]:
    return [get_lemma(word, moprh) for word in text]


def get_tonality(text: list[str], tone_features: pd.DataFrame) -> list[float | int]:
    tonalities = []
    for word in text:
        row = tone_features[tone_features["Words"] == word].values
        if row.size > 0:
            tonalities.append(row[0][1])
    if not tonalities:
        tonalities = [0]
    pos = neg = 0
    for tone in tonalities:
        if tone > 0:
            pos += 1
        elif tone < 0:
            neg += 1
    return [sum(tonalities) / len(tonalities), max(tonalities), min(tonalities), pos, neg]


morph = pymorphy3.MorphAnalyzer()
tone_mat = []

for sentence in tqdm(sentences):
    lemmatized = lemmatize(sentence, morph)
    tone_vec = get_tonality(lemmatized, lang_feat)
    tone_mat.append(tone_vec)

  0%|          | 0/226834 [00:00<?, ?it/s]

In [ ]:
np.save("tone_data.npy", np.array(tone_mat))
tone_mat[:4]

[[0.111111111111111, 0.333333333333333, 0.0, 1, 0],
 [0.5, 1.0, 0.0, 1, 0],
 [-0.833333333333333, -0.833333333333333, -0.833333333333333, 0, 1],
 [-0.51111111111111, 0.0, -1.33333333333333, 0, 2]]

# 4. Вытаскиваем морфологические тэги

In [ ]:
def get_tags(text: list[str], morph: pymorphy3.MorphAnalyzer, all_tags: list[str]) -> list[int]:
    tags = Counter([morph.parse(word)[0].tag.POS for word in text])
    return [tags.get(tag, 0) for tag in all_tags]


tag_mat = []
all_tags = list(morph.TagClass.PARTS_OF_SPEECH)
for sentence in tqdm(sentences):
    tag_vec = get_tags(sentence, morph, all_tags)
    tag_mat.append(tag_vec)

  0%|          | 0/226834 [00:00<?, ?it/s]

In [ ]:
np.save("tag_data.npy", np.array(tag_mat))
tag_mat[:4]

[[0, 0, 0, 3, 0, 0, 0, 1, 0, 0, 1, 1, 2, 0, 1, 0, 0],
 [0, 1, 0, 2, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1],
 [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 4, 0, 0, 4, 0, 0, 0, 1, 0, 3, 0, 0, 1, 0]]

# 5. TF-iDF

In [ ]:
sentences_coonected = [" ".join(text) for text in sentences]

vectorizer = TfidfVectorizer(min_df=0.0002, max_features=1000)
tf_idf = vectorizer.fit_transform(sentences_coonected)

In [ ]:
from scipy.sparse import save_npz

save_npz("vectors_sparse_tfidf.npz", tf_idf)

tf_idf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 788989 stored elements and shape (226834, 1000)>

# Check

In [ ]:
from scipy.sparse import load_npz

assert np.allclose(np.load("tone_data.npy"), np.array(tone_mat))
assert np.allclose(np.load("tag_data.npy"), np.array(tag_mat))
load_npz("vectors_sparse_tfidf.npz")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 788989 stored elements and shape (226834, 1000)>

In [ ]:
glue_sentences = [" ".join(sentence) for sentence in sentences]
with open("your_file.txt", "w") as f:
    for line in glue_sentences:
        f.write(f"{line}\n")